In [ ]:
import pandas as pd
import numpy as np
import os
import itertools as it
from snp_analysis_tools_sherlock import *
from coalescence_analysis_tools import *
from plot_tools import *
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:
all_dfs=[]
species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
e003_metadata = pd.read_csv('e003_with_passage_one_redo_good.csv').set_index('sample')
for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
     #   try:
        fname = f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/{sp}/{ino}_parent1_info.csv'
        if os.path.exists(fname):
            df_both=get_both_dfs(fname)#.set_index('sample')
       # except:
        #    continue
            df_both['species_id']=sp
            df_both['sp_plot']=df_metadata.loc[df_metadata['species_id']==int(sp),'species'].values[0]
         #   print(df_metadata.loc[df_metadata['species_id']==int(sp),'species'])
            
            df_both = df_both.loc[np.intersect1d(df_both.index.values,e003_metadata.index.values),:]
            df_both=df_both.loc[df_both['total_shift']<.1,:]
        
            df_both_meta = pd.concat([df_both,e003_metadata.loc[np.intersect1d(df_both.index.values,
                                                                                           e003_metadata.index.values),:]],
                                                 axis=1).reset_index()
            all_dfs.append(df_both_meta)
            
all_dfs=pd.concat(all_dfs)
all_dfs.head()

In [ ]:
full_dfp7= all_dfs.loc[all_dfs['passage']==7,:]
full_dfp0= all_dfs.loc[all_dfs['passage']==0,:]

full_dfp0.head()

In [ ]:
cmap_sp = {'s__Flavonifractor plautii': '#77AADD',
 's__Parabacteroides distasonis': '#EE8866',
 's__Bacteroides uniformis': '#FFAABB',
 's__Escherichia coli_D': '#EEDD88', 
 's__Bacteroides thetaiotaomicron': '#99DDFF',
 's__Dorea formicigenerans': '#44BB99',
 's__Parasutterella excrementihominis': '#BBCC33',
 's__Bacteroides_B dorei': '#AAAA00',
 'other': '#DDDDDD'}


## selection is predictable!!!

In [ ]:
from scipy.spatial.distance import jensenshannon
df_abundance = pd.read_csv('e003_coal_redo_good_abundances.csv').drop(columns='Unnamed: 0')
goodsp = df_abundance.groupby(['species_id']).max().reset_index()
goodsp = goodsp.loc[goodsp['relative_abundance']>0,'species_id'].values
df_abundance = df_abundance.loc[df_abundance['species_id'].isin(goodsp),:]
df_abundance['mesocosm-passage'] = df_abundance['mesocosm'] + '-' + df_abundance['passage'].astype(str)

df_abundance['relative_abundance'].min()

metadata = pd.read_csv('e003_with_passage_one_redo_good.csv')
metadatap7=metadata.loc[metadata['passage']==7,:]
samples= []
full_dfp7['dist_parent1'] = 0
full_dfp7['dist_parent2'] = 0
full_dfp7['dist_parent_inoculumn'] = 0
full_dfp7['med_dist_reps'] = 0
for sample in full_dfp7['sample'].unique():
    spog_df = df_abundance.loc[df_abundance['sample'] == sample,:].sort_values(by='species_id')
    mesocosm = spog_df['type_mesocosm'].unique()[0]
    parent_subject1, parent_subject2, parent_media, media = mesocosm.split('-')
    
    ins = metadata.loc[metadata['is_inoculumn'],:]
    ins = ins.loc[ins['parent_media'] == parent_media,:]
    ins1 = ins.loc[ins['parent_subjects'] == parent_subject1 + '-' +  parent_subject1,'sample'].values[0]
    ins2 = ins.loc[ins['parent_subjects'] == parent_subject2 + '-' +  parent_subject2,'sample'].values[0]
    ins3 = ins.loc[ins['parent_subjects'] == parent_subject1 + '-' +  parent_subject2,'sample'].values[0]
    sp1 = df_abundance.loc[df_abundance['sample'] == ins1,:].sort_values(by='species_id')
    sp2 = df_abundance.loc[df_abundance['sample'] == ins2,:].sort_values(by='species_id')
    sp3 = df_abundance.loc[df_abundance['sample'] == ins3,:].sort_values(by='species_id')

    JSD1 = jensenshannon(spog_df['relative_abundance'].values,sp1['relative_abundance'].values)
    JSD2 = jensenshannon(spog_df['relative_abundance'].values,sp2['relative_abundance'].values)
    JSD3 = jensenshannon(spog_df['relative_abundance'].values,sp3['relative_abundance'].values)
    other_samples = metadatap7.loc[metadatap7['type_mesocosm']==f'{parent_subject1}-{parent_subject2}-{parent_media}-{media}','sample'].values
    JSD4s=[]
    for samp in other_samples:
        if samp==sample:
            continue
        else:
            sp4 = df_abundance.loc[df_abundance['sample'] == samp,:].sort_values(by='species_id')
            JSD4 = jensenshannon(spog_df['relative_abundance'].values,sp4['relative_abundance'].values)
            JSD4s.append(JSD4)
  #  print(JSD4s)
    samples.append(sample)
    
    
  #  dist_parent1.append(JSD1)
   # dist_parent2.append(JSD2)
   # dist_inoculumn.append(JSD3) 

    full_dfp7.loc[full_dfp7['sample'] == sample,'dist_parent1'] = JSD1
    full_dfp7.loc[full_dfp7['sample'] == sample,'dist_parent2'] = JSD2
    full_dfp7.loc[full_dfp7['sample'] == sample,'dist_parent_inoculumn'] = JSD3
    full_dfp7.loc[full_dfp7['sample'] == sample,'med_dist_reps'] = np.median(JSD4s)

In [ ]:

e003_metadatap7 = e003_metadata.loc[e003_metadata['passage'] == 7,:]
e003_metadatap7['replicate'] = np.nan 
e003_metadatap7 = e003_metadatap7.sort_values(by='comm')
reps = [1,2,3,4]
full_dfp7['type_meso']=full_dfp7['type_mesocosm'].copy()
full_dfp7['species-type_meso'] = full_dfp7['species_id'].astype(str) + '-' + full_dfp7['type_meso']
full_dfp7['replicate'] = 0.
full_dfp0['type_meso']=full_dfp0['type_mesocosm']
full_dfp0['species-type_meso'] = full_dfp0['species_id'].astype(str) + '-' + full_dfp0['type_meso']
full_dfp0med=full_dfp0.groupby(['species-type_meso']).median(numeric_only=True).reset_index()
full_dfp7=full_dfp7.loc[full_dfp7['species-type_meso'].isin(full_dfp0med['species-type_meso'].values),:]

for species_type_meso in full_dfp7['species-type_meso'].unique():
    num_reps = len(full_dfp7.loc[full_dfp7['species-type_meso'] == species_type_meso, 'replicate'])
    full_dfp7.loc[full_dfp7['species-type_meso'] == species_type_meso, 'replicate'] = reps[:num_reps]
  #  print(full_dfp0med.loc[full_dfp0med['species-type_meso'] == species_type_meso, 'actual_med1'].values)
    full_dfp7.loc[full_dfp7['species-type_meso'] == species_type_meso, 'p0_med']= \
        full_dfp0med.loc[full_dfp0med['species-type_meso'] == species_type_meso, 'actual_med1'].values[0]
    


full_dfp7_sum = full_dfp7.groupby(['species-type_meso','type_meso']).sum().reset_index()
full_dfp7_sum_good = full_dfp7_sum.loc[full_dfp7_sum['replicate'] >=2,:]
good_type_mesos = full_dfp7_sum_good['species-type_meso'].values
full_dfp7_good = full_dfp7.loc[full_dfp7['species-type_meso'].isin(good_type_mesos ),:]

median_set = full_dfp7_good.groupby(['species-type_meso','type_meso',]).median(numeric_only=True).reset_index()

full_dfp7_good['med_strain_freq'] = full_dfp7_good['species-type_meso'].transform(lambda x: \
                                                              median_set.loc[median_set['species-type_meso']==x,'actual_med1'].values[0])



#full_dfp7_good['med_strain_shift'] = full_dfp7_good['species-type_meso'].transform(lambda x: \
 #                                                             median_set.loc[median_set['species-type_meso']==x,'shift_from_inoculumn'].values[0])


full_dfp7_good['abs_diff_from_med'] = full_dfp7_good['med_strain_freq']- full_dfp7_good['actual_med1']
full_dfp7_good['abs_diff_from_med'] = full_dfp7_good['abs_diff_from_med'].abs()

#full_dfp7_good['abs_diff_from_med_shift'] = full_dfp7_good['med_strain_shift']- full_dfp7_good['shift_from_inoculumn']
#full_dfp7_good['abs_diff_from_med_shift'] = full_dfp7_good['abs_diff_from_med_shift'].abs()
#full_dfp7_good['abs_diff_from_med_norm'] = full_dfp7_good['abs_diff_from_med']/full_dfp7_good['med_strain_freq']

p = iqplot.ecdf(full_dfp7_good,#.loc[full_dfp7_good['abs_diff_from_med']>0.,:],
                q= 'abs_diff_from_med', #height=400,
                     #rug=False,
                  )#,complementary =True)
#p.x_range = bokeh.models.Range1d(0,.5)
bokeh.io.show(p)

In [ ]:
p = iqplot.histogram(full_dfp7_good,#.loc[full_dfp7_good['abs_diff_from_med']>0.,:],
                q= 'abs_diff_from_med', #height=400,
                     rug=False,
                  )#,complementary =True)
#p.x_range = bokeh.models.Range1d(0,.5)
#bokeh.io.show(p)

frequencies, edges = np.histogram(full_dfp7_good['abs_diff_from_med'], 30)
#print('Values: %s, Edges: %s' % (frequencies.shape[0], edges.shape[0]))
hv.Histogram((edges, frequencies)).opts(width=500, height=200, xlabel='Abs Dist From Median', 
                                        ylabel='Counts',fill_color= bokeh.palettes.Bright[6][1]
                                       )

In [ ]:
full_dfp7_good['minor_strain_freq'] = full_dfp7_good['actual_med1'].copy()
full_dfp7_good.loc[full_dfp7_good['p0_med']>.5,'minor_strain_freq'] = 1.-full_dfp7_good.loc[full_dfp7_good['p0_med']>.5,'minor_strain_freq']

frequencies, edges = np.histogram(full_dfp7_good['minor_strain_freq'], 20)
#print('Values: %s, Edges: %s' % (frequencies.shape[0], edges.shape[0]))
hv.Histogram((edges, frequencies)).opts(width=500, height=200, xlabel='Final Frequency of Initially Minor Strain', 
                ylabel='Counts',fill_color= bokeh.palettes.Bright[6][1]
                                       )

In [ ]:
#full_dfp7_med = full_dfp7.groupby(['mesocosm','type_meso',]).median(numeric_only=True)
scatter1 = hv.Scatter(full_dfp7_good.groupby(['sp_plot','mesocosm','parent_subjects']).median(numeric_only=True).reset_index(), kdims =  'med_dist_reps', 
                      vdims=['abs_diff_from_med','sp_plot', ]).opts(#size=4, 
                                                                                 color = 'sp_plot',height=350,
                                                                                     # color='type_meso',
                                                                                       width = 800,#cmap=type_meso,
                                                                       line_color='black',
                                                                       size=8,

                                                                      
                                                                                      cmap = bokeh.palettes.Set3[11][::-1],
                                                                                                        colorbar=True, 
                                                                                                #   logx=True, logy=True,
                                                                                            legend_position='right',
                                                                                        xlim=(0.05,.45),ylim=(-0.01,.7),
                                                                            ylabel='Med(Diff Strain Freq from Med)',
                                                                      # logy=True,
                                                                                               xlabel='Med JSD from Other Reps',
                                                                                                  alpha = 1.,)
                                                                                             #   logx=True,
                                                                                              #  logy=True,
                                                                                            # xlim=(0.001,1.001),ylim=(0.001,1.001),
                                                                                              # xlabel='selection',
                                                                                               #ylabel='Dist from Median')
                                                                                                #    xlim=(-1.5, 1.5), ylim=(-1.5, 1.5))

scatter1

In [ ]:
plots=[]
for sp in full_dfp7_good['sp_plot'].unique():
    full_dfp7_goodsp=full_dfp7_good.loc[full_dfp7_good['sp_plot']==sp,:]
    cmap_parents = {'AA-AE':bokeh.palettes.Set3[11][-1],'AA-AF':bokeh.palettes.Set3[11][-2],
                    'AE-AF':bokeh.palettes.Set3[11][-3]}
    scatter1 = hv.Scatter(full_dfp7_goodsp.groupby(['species','mesocosm','parent_subjects']).median(numeric_only=True).reset_index(), kdims =  'med_dist_reps', 
                      vdims=['abs_diff_from_med', 'parent_subjects']).opts(#size=4, 
                                                                                 color = 'parent_subjects',#height=350,
                                                                                     # color='type_meso',
                                                                                       width = 400,#cmap=type_meso,
                                                                       line_color='black',
                                                                       size=8,

                                                                      
                                                                                      cmap = cmap_parents,
                                                                                                        colorbar=True, 
                                                                                                #   logx=True, logy=True,
                                                                                            legend_position='right',
                                                                                         title = sp,xlim=(0.1,.45),ylim=(-0.01,.7),
    
                                                                            ylabel='Med(Diff Strain Freq from Med)',
                                                                     #  logy=True,
                                                                                               xlabel='Med JSD from Other Reps',
                                                                                                  alpha = 1.,)
                                                                                             #   logx=True,
                                                                                              #  logy=True,
                                                                                            # xlim=(0.001,1.001),ylim=(0.001,1.001),
                                                                                              # xlabel='selection',
                                                                                               #ylabel='Dist from Median')
                                                                                                #    xlim=(-1.5, 1.5), ylim=(-1.5, 1.5))

    bokeh.io.show(hv.render(scatter1))
    plots.append(hv.render(scatter1))

In [ ]:
bokeh.io.export_png(bokeh.layouts.gridplot(plots,ncols=3),filename='JSDsp.png')

In [ ]:
full_df= all_dfs.copy()
#e003_metadatap7 = e003_metadata.loc[e003_metadata['passage'] == 7,:]
e003_metadata['replicate'] = np.nan 
e003_metadata = e003_metadata.sort_values(by='comm')
reps = [1,2,3,4]
full_df['type_meso']=full_df['type_mesocosm'].copy()
full_df['species-type_meso-passage'] = full_df['species_id'].astype(str) + '-' + full_df['type_meso'] + '-' + full_df['passage'].astype(str)
full_df['replicate'] = 0.

for species_type_meso in full_df['species-type_meso-passage'].unique():
    num_reps = len(full_df.loc[full_df['species-type_meso-passage'] == species_type_meso, 'replicate'])
    full_df.loc[full_df['species-type_meso-passage'] == species_type_meso, 'replicate'] = reps[:num_reps]
  #  print(full_dfp0med.loc[full_dfp0med['species-type_meso'] == species_type_meso, 'actual_med1'].values)



full_df_sum = full_df.groupby(['species-type_meso-passage','type_meso']).sum().reset_index()
full_df_sum_good = full_df_sum.loc[full_df_sum['replicate'] >=2,:]
good_type_mesos = full_df_sum_good['species-type_meso-passage'].values
full_df_good = full_df.loc[full_df['species-type_meso-passage'].isin(good_type_mesos ),:]

median_set = full_df_good.groupby(['species-type_meso-passage','type_meso',]).median(numeric_only=True).reset_index()

full_df_good['med_strain_freq'] = full_df_good['species-type_meso-passage'].transform(lambda x: \
                                                              median_set.loc[median_set['species-type_meso-passage']==x,'actual_med1'].values[0])



#full_dfp7_good['med_strain_shift'] = full_dfp7_good['species-type_meso'].transform(lambda x: \
 #                                                             median_set.loc[median_set['species-type_meso']==x,'shift_from_inoculumn'].values[0])


full_df_good['abs_diff_from_med'] = full_df_good['med_strain_freq']- full_df_good['actual_med1']
full_df_good['abs_diff_from_med'] = full_df_good['abs_diff_from_med'].abs()

#full_dfp7_good['abs_diff_from_med_shift'] = full_dfp7_good['med_strain_shift']- full_dfp7_good['shift_from_inoculumn']
#full_dfp7_good['abs_diff_from_med_shift'] = full_dfp7_good['abs_diff_from_med_shift'].abs()
#full_dfp7_good['abs_diff_from_med_norm'] = full_dfp7_good['abs_diff_from_med']/full_dfp7_good['med_strain_freq']

p = iqplot.ecdf(full_df_good,#.loc[full_dfp7_good['abs_diff_from_med']>0.,:],
                q= 'abs_diff_from_med', #height=400,
                     #rug=False,
                  )#,complementary =True)
#p.x_range = bokeh.models.Range1d(0,.5)
bokeh.io.show(p)

In [ ]:
p = iqplot.histogram(full_df_good,#.loc[full_dfp7_good['abs_diff_from_med']>0.,:],
                q= 'abs_diff_from_med', #height=400,
                     rug=False,
                  )#,complementary =True)
#p.x_range = bokeh.models.Range1d(0,.5)
#bokeh.io.show(p)

frequencies, edges = np.histogram(full_df_good['abs_diff_from_med'], 30)
#print('Values: %s, Edges: %s' % (frequencies.shape[0], edges.shape[0]))
hv.Histogram((edges, frequencies)).opts(width=500, height=200, xlabel='Abs Dist From Median', 
                                       # logy=True,
                                        ylabel='Counts',fill_color= bokeh.palettes.Bright[6][1]
                                       )

### group by inoculumn

In [ ]:
from scipy.spatial.distance import jensenshannon
df_abundance = pd.read_csv('e003_coal_redo_good_abundances.csv').drop(columns='Unnamed: 0')
goodsp = df_abundance.groupby(['species_id']).max().reset_index()
goodsp = goodsp.loc[goodsp['relative_abundance']>0,'species_id'].values
df_abundance = df_abundance.loc[df_abundance['species_id'].isin(goodsp),:]
df_abundance['mesocosm-passage'] = df_abundance['mesocosm'] + '-' + df_abundance['passage'].astype(str)


metadata = pd.read_csv('e003_with_passage_one_redo_good.csv')


metadatap7=metadata.loc[metadata['passage']==7,:]
samples= []

pairwise_JSDs = []
sample1s=[]
sample2s=[]
same_in_samples = []
type_mesos = []

for type_meso in full_dfp7['type_mesocosm'].unique():
    df_type_meso = full_dfp7.loc[full_dfp7['type_mesocosm']==type_meso,:]
    samples = df_type_meso['sample'].unique()
    for s1,s2 in it.combinations(samples,2):
        in_sample1=df_type_meso.loc[df_type_meso['sample']==s1,'inoculumn_sample'].unique()
        if len(in_sample1)<1:
            continue
        in_sample2=df_type_meso.loc[df_type_meso['sample']==s2,'inoculumn_sample'].unique()
        if len(in_sample1)<1:
            continue
        if in_sample1[0]==in_sample2[0]:
            same_in_samples.append(True)
        else:
            same_in_samples.append(False)
        sample1s.append(s1)
        sample2s.append(s2)
        type_mesos.append(type_meso)
        sp1 = df_abundance.loc[df_abundance['sample'] == s1,:].sort_values(by='species_id')
        sp2 = df_abundance.loc[df_abundance['sample'] == s2,:].sort_values(by='species_id')
        JSD1 = jensenshannon(sp1['relative_abundance'].values,sp2['relative_abundance'].values)
        pairwise_JSDs.append(JSD1)
        

In [ ]:
df_pairwise_JSD = pd.DataFrame(data={'pairwise_JSDs':pairwise_JSDs,'sample1':sample1s,
                                     'sample2':sample2s,'same_in_samples':same_in_samples,
                                     'type_meso':type_mesos})
df_pairwise_JSD.min()                                    

In [ ]:

e003_metadatap7 = e003_metadata.loc[e003_metadata['passage'] == 7,:]
df_abundance_p7=df_abundance.loc[df_abundance['passage']==7,:]
df_abundance_p7=df_abundance_p7.loc[df_abundance_p7['sample'].isin(e003_metadatap7.index.values),:]
e003_metadatap7['replicate'] = np.nan 

e003_metadatap7 = e003_metadatap7.sort_values(by='comm')
reps = [1,2,3,4]
full_dfp7['type_meso']=full_dfp7['type_mesocosm'].copy()
full_dfp7['species-type_meso'] = full_dfp7['species_id'].astype(str) + '-' + full_dfp7['type_meso']
df_abundance_p7['species-type_meso']=df_abundance_p7['species_id'].astype(str) + '-' + df_abundance_p7['type_mesocosm']
full_dfp7['replicate'] = 0.
speciestype_mesos = []
sample1s=[]
sample2s=[]
JSDs=[]
dffs_freq=[]
same_in_samples=[]
s1_abuns=[]
s2_abuns=[]
for species_type_meso in full_dfp7['species-type_meso'].unique():
    num_reps = len(full_dfp7.loc[full_dfp7['species-type_meso'] == species_type_meso, 'replicate'])
    full_dfp7.loc[full_dfp7['species-type_meso'] == species_type_meso, 'replicate'] = reps[:num_reps]
    if num_reps<2:
        continue 
    df_sp_type_meso=full_dfp7.loc[full_dfp7['species-type_meso']==species_type_meso,:]
    df_abun_sp_type_meso=df_abundance_p7.loc[df_abundance_p7['species-type_meso']==species_type_meso,:]
    for s1,s2 in it.combinations(df_sp_type_meso['sample'].unique(),2):
        if len(df_pairwise_JSD.loc[(df_pairwise_JSD['sample1']==s1)*(df_pairwise_JSD['sample2']==s2),:])>0:
            
            df_s1s2=df_pairwise_JSD.loc[(df_pairwise_JSD['sample1']==s1)*(df_pairwise_JSD['sample2']==s2),:]
            s1_abun=df_abun_sp_type_meso.loc[df_abun_sp_type_meso['sample']==s1,'relative_abundance'].values[0]
            sample1s.append(s1)
            s2_abun=df_abun_sp_type_meso.loc[df_abun_sp_type_meso['sample']==s2,'relative_abundance'].values[0]
            sample2s.append(s2)
        else:
            df_s1s2=df_pairwise_JSD.loc[(df_pairwise_JSD['sample1']==s2)*(df_pairwise_JSD['sample2']==s1),:]
            s1_abun=df_abun_sp_type_meso.loc[df_abun_sp_type_meso['sample']==s2,'relative_abundance'].values[0]
            s2_abun=df_abun_sp_type_meso.loc[df_abun_sp_type_meso['sample']==s1,'relative_abundance'].values[0]
            sample1s.append(s2)
            sample2s.append(s1)
        same_in_samples.append(df_s1s2['same_in_samples'].values[0])
        s1_abuns.append(s1_abun)
        s2_abuns.append(s2_abun)
        JSDs.append(df_s1s2['pairwise_JSDs'].values[0])
        
       # diff=np.abs(df_sp_type_meso.loc[df_sp_type_meso['sample']==s1,'actual_med1']-df_sp_type_meso.loc[df_sp_type_meso['sample']==s2,'actual_med1'])
        diff = df_sp_type_meso.loc[df_sp_type_meso['sample']==s1,'actual_med1'].values[0]-df_sp_type_meso.loc[df_sp_type_meso['sample']==s2,'actual_med1'].values[0]
        dffs_freq.append(np.abs(diff))
        speciestype_mesos.append(species_type_meso)

        
  #  print(full_dfp0med.loc[full_dfp0med['species-type_meso'] == species_type_meso, 'actual_med1'].values)




In [ ]:

df_pairwise_JSD_freq = pd.DataFrame(data={'pairwise_JSDs':JSDs,'diff_freq':dffs_freq, 'sample1':sample1s,
                                     'sample2':sample2s,'same_in_samples':same_in_samples,
                                    #'fold_change_focal_sp': XX,
                                          's1_abuns':s1_abuns, 's2_abuns':s2_abuns,
                                     'species-type_meso':speciestype_mesos})
df_pairwise_JSD_freq['species_id']=df_pairwise_JSD_freq['species-type_meso'].transform(lambda x: x.split('-')[0])
df_pairwise_JSD_freq['sp_plot']=df_pairwise_JSD_freq['species_id'].transform(lambda x: df_metadata.loc[df_metadata['species_id']==int(x),'species'].values[0])


In [ ]:
df_pairwise_JSD_freq['log_fold_change']=np.log(df_pairwise_JSD_freq['s1_abuns'])-np.log(df_pairwise_JSD_freq['s2_abuns'])
df_pairwise_JSD_freq['log_fold_change']

In [ ]:
plots=[]
for sp in full_dfp7_good['sp_plot'].unique():
    full_dfp7_goodsp=df_pairwise_JSD_freq.loc[df_pairwise_JSD_freq['sp_plot']==sp,:]
    cmap_parents = {True:bokeh.palettes.Set3[11][-1],False:bokeh.palettes.Set3[11][-2]}
    scatter1 = hv.Scatter(full_dfp7_goodsp.sort_values(by='same_in_samples'), kdims =  'pairwise_JSDs', 
                      vdims=['diff_freq', 'same_in_samples']).opts(#size=4, 
                                                                                 color = 'same_in_samples',#height=350,
                                                                                     # color='type_meso',
                                                                                       width = 400,#cmap=type_meso,
                                                                       line_color='black',
                                                                       size=8,

                                                                      
                                                                                      cmap = cmap_parents,
                                                                                                        colorbar=True, 
                                                                                                #   logx=True, logy=True,
                                                                                            legend_position='right',
                                                                                         title = sp,xlim=(0.05,.5),ylim=(-0.01,.7),
    
                                                                            ylabel='Pairwise Abs Diff Strain Freq',
                                                                     #  logy=True,
                                                                                               xlabel='Pairwise JSD',
                                                                                                  alpha = 1.,)
                                                                                             #   logx=True,
                                                                                              #  logy=True,
                                                                                            # xlim=(0.001,1.001),ylim=(0.001,1.001),
                                                                                              # xlabel='selection',
                                                                                               #ylabel='Dist from Median')
                                                                                                #    xlim=(-1.5, 1.5), ylim=(-1.5, 1.5))

    bokeh.io.show(hv.render(scatter1))
    plots.append(hv.render(scatter1))

In [ ]:
df_pairwise_JSD_freq['abs_log_fold_change']=df_pairwise_JSD_freq['log_fold_change'].abs()

In [ ]:
scatter1 = hv.Scatter(df_pairwise_JSD_freq.sort_values(by='abs_log_fold_change'), kdims =  'pairwise_JSDs', 
                      vdims=['diff_freq', 'abs_log_fold_change']).opts(#size=4, 
                                                                                 color = 'abs_log_fold_change',#height=350,
                                                                                     # color='type_meso',
        
                                                                                       width = 500,#cmap=type_meso,
                                                                     #  height=400,
    line_color='black',alpha=1.,
                                                                       size=8,

                                                                      
                                                                                     # cmap =  bokeh.palettes.Set3[11][::-1],
                                                                                                        colorbar=True, 
                                                                                                #   logx=True, logy=True,
                                                                                            legend_position='right',
                                                                                      ylim=(-0.01,.7),
    
                                                                            ylabel='Pairwise Abs Diff Strain Freq',
                                                                     #  logx=True,
                                                                                               xlabel='Pairwise JSD',
                                                                                                )
                                                                                             #   logx=True,
                                                                                              #  logy=True,
                                                                                            # xlim=(0.001,1.001),ylim=(0.001,1.001),
                                                                                              # xlabel='selection',
                                                                                               #ylabel='Dist from Median')
                                                                                                #    xlim=(-1.5, 1.5), ylim=(-1.5, 1.5))

bokeh.io.show(hv.render(scatter1))
plots.append(hv.render(scatter1))

In [ ]:
from scipy.stats import kstest
rvs1=df_pairwise_JSD_freq.loc[df_pairwise_JSD_freq['pairwise_JSDs']<=.24,'diff_freq'].values
rvs2=df_pairwise_JSD_freq.loc[df_pairwise_JSD_freq['pairwise_JSDs']>.24,'diff_freq'].values
kstest(rvs1, rvs2)

In [ ]:
bokeh.io.export_png(bokeh.layouts.gridplot(plots,ncols=3),filename='JSDsp_by_in.png')

In [ ]:
p = iqplot.histogram(df_pairwise_JSD_freq,cats='same_in_samples',#.loc[full_dfp7_good['abs_diff_from_med']>0.,:],
                q= 'diff_freq', #height=400,
                     rug=False,arrangement ='overlay',density=True,
                  )#,complementary =True)
#p.x_range = bokeh.models.Range1d(0,.5)
bokeh.io.show(p)



In [ ]:
p = iqplot.ecdf(df_pairwise_JSD_freq,cats='same_in_samples',#.loc[full_dfp7_good['abs_diff_from_med']>0.,:],
                q= 'diff_freq', #height=400,
                   #  rug=False,arrangement ='overlay',density=True,
                  )#,complementary =True)
#p.x_range = bokeh.models.Range1d(0,.5)
p.xaxis.axis_label='Abs Diff in Strain Freq'
bokeh.io.show(p)

In [ ]:
p = iqplot.histogram(df_pairwise_JSD_freq,cats='same_in_samples',#.loc[full_dfp7_good['abs_diff_from_med']>0.,:],
                q= 'pairwise_JSDs', #height=400,
                     rug=False,arrangement ='overlay',density=True,
                  )#,complementary =True)
#p.x_range = bokeh.models.Range1d(0,.5)
bokeh.io.show(p)

In [ ]:
p = iqplot.ecdf(df_pairwise_JSD_freq,cats='same_in_samples',#conf_int=True,#.loc[full_dfp7_good['abs_diff_from_med']>0.,:],
                q= 'pairwise_JSDs', #height=400,
                   
                  )#,complementary =True)
#p.x_range = bokeh.models.Range1d(0,.5)
p.xaxis.axis_label='Pairwise JSD'
bokeh.io.show(p)

In [ ]:
full_dfp7_good.columns.values

In [ ]:
cmap_parents = {True:bokeh.palettes.Set3[11][-1],False:bokeh.palettes.Set3[11][-2]}
scatter1 = hv.Scatter(df_pairwise_JSD_freq.groupby(['sample1','sample2','same_in_samples']).max(numeric_only=True).reset_index(), kdims =  'pairwise_JSDs', 
                      vdims=['diff_freq', 'same_in_samples']).opts(#size=4, 
                                                                                 color = 'same_in_samples',#height=350,
                                                                                     # color='type_meso',
                                                                                       width = 400,#cmap=type_meso,
                                                                       line_color='black',
                                                                       size=8,#logy=True,

                                                                      
                                                                                      cmap = cmap_parents,
                                                                                                        colorbar=True, 
                                                                                                #   logx=True, logy=True,
                                                                                            legend_position='right',
                                                                                         title = sp,#xlim=(0.05,.5),ylim=(-0.01,.7),
    
                                                                            ylabel='Abs Diff Strain Freq',
                                                                     #  logy=True,
                                                                                               xlabel='Pairwise JSD',
                                                                                                  alpha = .5,)

In [ ]:
scatter1